# NB 3.3 &mdash; Variables categòriques: ordre, one-hot i categories minoritàries

**MP 5134** &mdash; UT3

*Dades: directori d'empreses amb activitat econòmica de les Illes Balears
(IBESTAT, 2024).*

---
### Què farem avui

Al NB 3.2 vam convertir en números una sola columna de text, la zona de
California Housing, que tenia cinc valors. Avui canviem de dades i les trobarem
**totes de text**: el directori de les empreses de les Balears. Hi veurem els
tres casos que es troben a la pràctica i com tractem cadascún d'ells.

És un notebook curt. Quan acabis hauries de saber respondre aquestes preguntes:

- Quan té sentit convertir les categories en 0, 1, 2, 3…, i quan no?
- Què passa si fem un one-hot d'una columna amb desenes de categories?
- Com s'agrupen les categories que surten poc, i qui decideix el llindar?

Pel camí trobaràs dues pauses **Ara tu**.

## 1. Les dades: quaranta-vuit mil empreses de les Illes

L'**IBESTAT**, l'institut d'estadística de les Illes Balears, publica cada any el
directori de les empreses que tenen activitat econòmica a les Illes, amb
llicència **CC-BY**: el pots fer servir per a qualsevol cosa, sempre que en citis
la font. Nosaltres en farem servir l'edició de 2024.

El fitxer original porta el **NIF i el nom de cada empresa**. Els hem tret abans de
penjar-lo, perquè no els necessitem, i és un bon costum: de les dades, agafa
només el que facis servir. A més, fixa't que al directori només hi ha
**societats i entitats**, no autònoms. No és casualitat: el NIF d'un autònom és el
seu DNI, i el que se'n sap és informació d'una persona concreta, protegida per la
llei de protecció de dades. Les dades d'una societat no ho són.

In [11]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Directori d'empreses amb activitat econòmica de les Illes Balears (IBESTAT, 2024, CC-BY)
URL_DADES = ("https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/"
             "refs/heads/main/UT03-Correlacions_preparacio_variables/ibestat_empreses/empreses_balears.csv")

df = pd.read_csv(URL_DADES)

print(f"Empreses: {len(df)}")
df.head()

Empreses: 48326


,illa,municipi,sector,forma_juridica,assalariats
0,Mallorca,Sineu,Transport i emmagatzematge,Societat anònima,D'1 a 9
1,Mallorca,Palma,Construcció,Societat anònima,Sense assalariats
2,Mallorca,Calvià,Hostaleria,Societat anònima,D'1 a 9
3,Mallorca,Palma,Energia elèctrica i gas,Societat anònima,Sense assalariats
4,Mallorca,Calvià,Activitats artístiques i d'oci,Societat anònima,Sense assalariats


Cada fila és una empresa, i les cinc columnes són de text:

| Columna | Què és |
|---|---|
| `illa` | l'illa on té la seu |
| `municipi` | el municipi on té la seu |
| `sector` | l'activitat principal, segons la classificació oficial (CNAE) |
| `forma_juridica` | societat limitada, anònima, cooperativa… |
| `assalariats` | el tram de treballadors assalariats que té |

Comptem quants valors diferents té cada una.

In [12]:
df.nunique()

illa               4
municipi          67
sector            19
forma_juridica    16
assalariats        5
dtype: int64

Des de 4 illes fins a 67 municipis. Encara no sabem quin model entrenarem amb aquestes
dades però la regla del
NB 3.2 no canvia: **les fàbriques (estimators) aprenen només de l'entrenament**. Així que, com
sempre, abans de res dividim les dades fent els splits.

In [13]:
train, prova = train_test_split(df, test_size=0.2, random_state=42)

print(f"Empreses d'entrenament: {len(train)}")
print(f"Empreses de prova:      {len(prova)}")

Empreses d'entrenament: 38660
Empreses de prova:      9666


## 2. Categories amb ordre: la codificació ordinal

Comencem pel tram d'assalariats.

In [14]:
train["assalariats"].value_counts()

assalariats
D'1 a 9              21166
Sense assalariats    13731
De 10 a 49            3273
De 50 a 199            407
200 o més               83
Name: count, dtype: int64

Aquesta columna és text, però és diferent de la zona de California Housing:
**té un ordre de veritat**. Una empresa de 10 a 49 assalariats és més gran que una
d'1 a 9, i més petita que una de 50 a 199. Quan les categories tenen un ordre,
el més natural és **numerar-les seguint aquest ordre**: 0, 1, 2, 3, 4. Així el model
sap que el tram 3 està entre el 2 i el 4. És la **codificació ordinal**, i la fa
la fàbrica `OrdinalEncoder`.

Hi ha un detall que no es pot oblidar: **l'ordre li hem de dir nosaltres**. La
fàbrica no sap llegir «de 10 a 49», i si no li diem res, ordenarà les categories
per ordre alfabètic.

In [15]:
from sklearn.preprocessing import OrdinalEncoder

ORDRE_ASSALARIATS = ["Sense assalariats", "D'1 a 9", "De 10 a 49", "De 50 a 199", "200 o més"]

fabrica_assalariats = OrdinalEncoder(categories=[ORDRE_ASSALARIATS])
fabrica_assalariats.fit(train[["assalariats"]])

exemple = train[["assalariats"]].drop_duplicates()   # una empresa de cada tram
exemple.assign(codi=fabrica_assalariats.transform(exemple))

,assalariats,codi
38199,Sense assalariats,0.0
12505,D'1 a 9,1.0
37238,De 10 a 49,2.0
8339,De 50 a 199,3.0
47757,200 o més,4.0


`categories=[ORDRE_ASSALARIATS]` li dona l'ordre. Va dins d'una llista perquè la
fàbrica pot codificar diverses columnes alhora, i espera una llista d'ordres, un
per columna. `assign` afegeix una columna a la taula només per mostrar-la, al costat
de l'original.

Ara una empresa sense assalariats és un 0 i una de més de 200 és un 4: **una sola
columna de números** que conserva l'ordre.

---
### Ara tu · Exercici 1: i si no li diem l'ordre?

Fes una fàbrica `OrdinalEncoder()` **sense** el paràmetre `categories`, prepara-la
amb `train[["assalariats"]]` i mira quin número dona a cada tram. Pots veure
l'ordre que ha triat amb `fabrica.categories_`.

Quin número rep una empresa sense assalariats? I una de 200 o més? Per què un model
que aprengués amb aquests números s'equivocaria?

Pista: aplica la fàbrica a la taula `exemple` de la cel·la anterior i compara els
dos codis, un al costat de l'altre.

In [16]:
# Escriu aquí el teu codi

## 3. Categories sense ordre: el one-hot de sempre

L'illa no té cap ordre: Mallorca no és «més» que Menorca. És el cas de la zona de
California Housing, i la recepta és la mateixa del NB 3.2, el one-hot: una columna
per illa.

In [17]:
from sklearn.preprocessing import OneHotEncoder

fabrica_illa = OneHotEncoder(sparse_output=False)
fabrica_illa.fit(train[["illa"]])

pd.DataFrame(fabrica_illa.transform(train[["illa"]]),
             columns=fabrica_illa.get_feature_names_out(), index=train.index).sum()

illa_Eivissa        6205.0
illa_Formentera      469.0
illa_Mallorca      28622.0
illa_Menorca        3364.0
dtype: float64

Quatre columnes, i la més petita, Formentera, té gairebé cinc-centes empreses a
l'entrenament. Cap problema: amb quatre categories i totes ben representades, el
one-hot és perfecte.

## 4. Moltes categories: agrupar les minoritàries

El municipi també és una categoria sense ordre. Però té un problema que l'illa no
té.

In [18]:
empreses_per_municipi = train["municipi"].value_counts()

print(f"Municipis diferents: {len(empreses_per_municipi)}")
print()
print("Els tres amb més empreses:")
print(empreses_per_municipi.head(3).to_string())
print()
print("Els tres amb menys:")
print(empreses_per_municipi.tail(3).to_string())

Municipis diferents: 67

Els tres amb més empreses:
municipi
Palma      14483
Eivissa     2708
Calvià      2042

Els tres amb menys:
municipi
Ariany        17
Escorca       12
Estellencs     8


Seixanta-set municipis, i molt desiguals: **Palma té més de 14.000 empreses a
l'entrenament, i Estellencs, 8**. Un one-hot tal qual faria 67 columnes, i moltes
serien columnes plenes de zeros i molt pocs uns. Un model no pot aprendre res de fiar
sobre Estellencs a partir de vuit empreses, i cada columna de més és feina i soroll.

La solució és la que ja vam fer servir amb les illes de Califòrnia, però ara a una
altra escala: **agrupar els municipis petits en una sola categoria, «altres»**,
amb `min_frequency`.

In [19]:
fabrica_municipi = OneHotEncoder(min_frequency=200, handle_unknown="infrequent_if_exist",
                                 sparse_output=False)
fabrica_municipi.fit(train[["municipi"]])

columnes_municipi = fabrica_municipi.get_feature_names_out()
agrupats = fabrica_municipi.infrequent_categories_[0]

print(f"Columnes del one-hot: {len(columnes_municipi)}")
print(f"Municipis agrupats a «altres»: {len(agrupats)}, "
      f"amb {empreses_per_municipi[agrupats].sum()} empreses en total")

Columnes del one-hot: 32
Municipis agrupats a «altres»: 36, amb 2939 empreses en total


Amb `min_frequency=200`, els municipis amb menys de 200 empreses a l'entrenament
van a parar a la columna comuna. Queden **31 municipis amb columna pròpia i una
columna d'«altres»**, on hi ha els 36 municipis petits, amb menys de 3.000
empreses entre tots. Hem passat de 67 columnes a 32, i hem perdut molt poca
informació. `infrequent_categories_` és la llista dels que ha agrupat, per si la
vols mirar.

D'on surt el 200? **De cap fórmula: és una decisió nostra.** Un llindar més alt dona
menys columnes, però tracta igual més municipis; un de més baix, al revés. És un
hiperparàmetre de la preparació, i, com tots, el triarem a la vista del que doni
el model amb la validació.

---
### Ara tu · Exercici 2: la forma jurídica

Fes el mateix amb la columna `forma_juridica`:

- Mira quantes empreses hi ha de cada forma a l'entrenament. Quantes formes
  tenen menys de 100 empreses? N'hi ha alguna que en tingui una de sola?
- Tria un `min_frequency` que et sembli raonable i fes-ne el one-hot. Quantes
  columnes en surten? Quines formes han quedat agrupades?
- Explica en una frase per què has triat aquest llindar.

Pista: és la cel·la anterior canviant `municipi` per `forma_juridica`.

In [20]:
# Escriu aquí el teu codi

## 5. Tot junt: una taula de números

Ara que tenim una recepta per a cada cas, preparem la taula sencera. Afegim una
última fàbrica per al sector, que té 19 categories i un parell de minoritàries,
i ajuntem el resultat de les quatre.

Ho farem en dos passos: primer, una funció que converteix **una** columna; després,
una altra que les converteix **totes**.

In [21]:
fabrica_sector = OneHotEncoder(min_frequency=100, handle_unknown="infrequent_if_exist",
                               sparse_output=False)
fabrica_sector.fit(train[["sector"]])


def convertir(taula, columna, fabrica):
    """Passa una columna de la taula per una fàbrica ja preparada i en torna el resultat com a taula."""
    numeros = fabrica.transform(taula[[columna]])
    return pd.DataFrame(numeros, columns=fabrica.get_feature_names_out(), index=taula.index)


convertir(train, "illa", fabrica_illa).head()

,illa_Eivissa,illa_Formentera,illa_Mallorca,illa_Menorca
38199,0.0,0.0,0.0,1.0
12505,0.0,0.0,1.0,0.0
773,0.0,0.0,1.0,0.0
36123,0.0,0.0,1.0,0.0
3394,0.0,0.0,1.0,0.0


La funció `convertir` és la mateixa idea que la funció `zona` del NB 3.2, però
més general. `zona` només sabia fer una cosa: passar la columna `ocean_proximity`
per `fabrica_zona`. `convertir` serveix per a qualsevol columna i qualsevol
fàbrica, perquè les rep com a **paràmetres**:

- `taula`: la taula d'on agafem la columna (`train` o `prova`);
- `columna`: el nom de la columna que volem convertir;
- `fabrica`: la fàbrica que l'ha de convertir, una de les que ja hem preparat amb `fit`.

Dins la funció, `fabrica` no és cap objecte nou: és **el nom que pren la fàbrica
que li passem** en cridar-la. A l'exemple, `convertir(train, "illa", fabrica_illa)`,
dins la funció `fabrica` és `fabrica_illa`, i `columna` és `"illa"`.

Les dues línies de dins ja les coneixes: `transform` converteix la columna en
números, i `pd.DataFrame` els posa en una taula amb el nom de cada columna
(`get_feature_names_out`) i les mateixes files que la taula original (`index`).
El resultat són les quatre columnes one-hot de les illes de la secció 3.

In [22]:
def preparar(taula):
    """Converteix les quatre columnes de text de la taula en números."""
    return pd.concat([convertir(taula, "assalariats", fabrica_assalariats),
                      convertir(taula, "illa",        fabrica_illa),
                      convertir(taula, "municipi",    fabrica_municipi),
                      convertir(taula, "sector",      fabrica_sector)], axis=1)


train_preparat = preparar(train)
prova_preparat = preparar(prova)

print(f"Entrenament: {train.shape[1]} columnes de text -> {train_preparat.shape[1]} columnes de números")
print(f"Prova:       {prova.shape[1]} columnes de text -> {prova_preparat.shape[1]} columnes de números")
train_preparat.head()

Entrenament: 5 columnes de text -> 55 columnes de números
Prova:       5 columnes de text -> 55 columnes de números


,assalariats,illa_Eivissa,illa_Formentera,illa_Mallorca,illa_Menorca,municipi_Alaior,municipi_Alcúdia,municipi_Andratx,municipi_Artà,municipi_Binissalem,...,sector_Construcció,sector_Educació,sector_Energia elèctrica i gas,sector_Finances i assegurances,sector_Hostaleria,sector_Indústria manufacturera,sector_Informació i comunicacions,sector_Sanitat i serveis socials,sector_Transport i emmagatzematge,sector_infrequent_sklearn
38199,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12505,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
773,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
36123,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3394,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


La funció `preparar` crida `convertir` quatre vegades, una per columna, cada
vegada amb la seva fàbrica, i enganxa els quatre resultats un al costat de l'altre
amb `pd.concat`, com fèiem al NB 3.2.

La cridem igual per a l'entrenament i per a la prova, i totes dues taules surten
amb **les mateixes 55 columnes**: 1 del tram d'assalariats, 4 de les illes, 32 dels
municipis i 18 dels sectors. Que coincideixin no és casualitat: totes les
fàbriques s'han preparat amb `train`, i converteixen qualsevol taula a la mateixa
forma. (La forma jurídica no hi és: és la teva de l'exercici 2.)

Ara aquesta taula ja la podria fer servir qualsevol model de scikit-learn.

## 6. El que ens emportem

Tres casos, tres receptes:

| La columna és… | Exemple | Recepta |
|---|---|---|
| categòrica **amb ordre** | tram d'assalariats | `OrdinalEncoder`, **dient-li l'ordre** |
| categòrica **sense ordre**, poques categories | illa, zona de Califòrnia | `OneHotEncoder` |
| categòrica **sense ordre**, moltes categories o algunes de molt petites | municipi, sector | `OneHotEncoder` amb `min_frequency` |

I la regla de tota la UT3 continua valent: les fàbriques fan el `fit` amb
l'entrenament i el `transform` amb tot el que vingui, i el paràmetre
`handle_unknown` les prepara per a categories que no han vist mai.

Amb això tanquem la part guiada de la UT3. A la consolidació ho faràs tot sol:
correlacions, preparació de variables i la fitxa de variables del teu model.